# Pixel Art Animation Retargeting — notebook Colab

Orchestre le projet en respectant la separation CPU / GPU imposee par le
quota Colab gratuit : la preparation des donnees et les sanity checks
tournent sur un runtime **CPU**. Ne passer en runtime **GPU** que pour
l'entrainement (jalon 3+).

Jalons couverts par ce notebook : **1 (donnees + sanity check)**,
**2 (modele + forward pass CPU)**, **3 (overfit volontaire)**,
**4 (scaling + entrainement complet, runtime GPU)**.

## 0. Setup (CPU)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/YuryTheYety/Animation_pixel_art.git'
BRANCH = 'claude/pixel-art-retargeting-ml-7o275r'  # passer a 'main' une fois la branche mergee

import os
if not os.path.isdir('/content/Animation_pixel_art'):
    !git clone --branch {BRANCH} {REPO_URL} /content/Animation_pixel_art
%cd /content/Animation_pixel_art
!git pull

In [ ]:
!pip install -q -r requirements.txt

## 1. Jalon 1 — Donnees + sanity check (CPU)

`data_prep.py` telecharge une seule fois les spritesheets LPC et sauvegarde
les paires pretraitees (reference RGBA, carte de pose niveaux de gris,
cible RGBA) sur Google Drive. Relancer cette cellule ne refait rien si le
manifest existe deja (idempotent) — utiliser `--force` pour tout regenerer.

In [ ]:
# --limit pour un test rapide (ex: 25 personnages). Retirer --limit pour le run complet.
!python data_prep.py --limit 25 --workers 12

In [ ]:
!python sanity_check.py --n 5 --seed 0

In [ ]:
import config
from IPython.display import Image as IPImage, display
display(IPImage(filename=config.SANITY_CHECK_OUTPUT))

**Verification humaine attendue ici avant de passer au jalon suivant :**
- les colonnes `reference` et `cible` montrent bien le meme personnage (memes couleurs)
- la colonne `pose` (grise, sans identite) correspond a la silhouette de `cible`
- pas de fond opaque residuel, pas de decalage entre la pose et la cible

## 2. Jalon 2 — Modele + forward pass (CPU)

`models.py` definit `UNetGenerator` (entree = concat([pose_map, reference]))
et `PatchDiscriminator` (conditionnel). Le smoke test verifie les shapes ET
que le conditionnement est bien cable canal par canal (pas juste les shapes).

In [ ]:
!python models.py

## 3. Jalon 3 — Overfit volontaire (passer en runtime **GPU** ici)

Objectif : verifier que le modele peut apprendre a recopier correctement
~20 personnages avant de scaler. Si l'overfit echoue, le bug est dans les
donnees / le conditionnement, pas dans le modele — corriger avant de
continuer.

`train.py` est resume-first : relancer la cellule reprend automatiquement
au dernier checkpoint (`data/checkpoints/last.pt`) au lieu de redemarrer a zero.

In [ ]:
# ~20 personnages, poses limitees pour un premier signal rapide sur T4.
# Retirer --num-poses pour utiliser toutes les poses des 20 personnages.
!python train.py --num-characters 20 --steps 3000 --batch-size 16

In [ ]:
import os
import config
from IPython.display import Image as IPImage, display
display(IPImage(filename=os.path.join(config.PREPARED_DIR, 'train_preview.png')))

**Verification humaine attendue :** la colonne `generee` doit ressembler
tres fortement a `cible` (memes couleurs, meme pose). Si l'IoU/l'aspect
visuel est mauvais apres plusieurs milliers de steps, revenir au jalon 1/2
avant de scaler — ne pas continuer sur un pipeline casse.

## 4. Jalon 4 — Scaling (dataset complet + entrainement long, runtime **GPU**)

Prepare tout le dataset disponible (pas de `--limit`) puis lance
l'entrainement sur l'integralite des personnages/poses, sans restriction.
Sessions Colab gratuites coupant, `train.py` reprend automatiquement au
dernier checkpoint : relancer la cellule d'entrainement autant de fois que
necessaire, elle continue toujours a partir de `last.pt`.

In [ ]:
# Dataset complet (tous les personnages/poses disponibles). CPU, a lancer
# une seule fois -- idempotent (ne refait rien si le manifest existe deja).
!python data_prep.py --workers 12

In [ ]:
# Runtime GPU requis ici. Relancer cette cellule reprend automatiquement
# (resume-first) au dernier checkpoint sur Drive -- pas besoin de tout
# refaire si la session Colab coupe.
!python train.py --steps 100000 --batch-size 16

## 5. Generation / test — animer un sprite (CPU ou GPU, peu importe)

`generate.py` fait uniquement de l'inference (aucun entrainement) : il
prend UNE image de reference et enchaine les poses d'une meme ligne de la
grille (= une animation/direction) avec le checkpoint deja entraine.

Upload ton sprite ici (glisser-deposer dans le panneau "Fichiers" a gauche,
ou executer la cellule d'upload ci-dessous), puis choisis une ligne
d'animation avec `--list-rows`.

In [ ]:
from google.colab import files
uploaded = files.upload()  # choisir ton sprite (idealement RGBA, proche de 64x64)
sprite_path = list(uploaded.keys())[0]
print(sprite_path)

In [ ]:
!python generate.py --list-rows

In [ ]:
# Ajuste ROW selon --list-rows ci-dessus. --checkpoint best (par defaut) ou last.
ROW = 8  # ex: souvent le cycle de marche selon la longueur de ligne (9 frames)
!python generate.py --reference-image "{sprite_path}" --row {ROW} --out data/prepared/test_gif.gif
!python generate.py --reference-image "{sprite_path}" --row {ROW} --out data/prepared/test_sheet.png

In [ ]:
from IPython.display import Image as IPImage, display
display(IPImage(filename="data/prepared/test_gif.gif"))
display(IPImage(filename="data/prepared/test_sheet.png"))

**Rappel :** le generateur reste calibre sur la distribution LPC (proportions,
cadrage, pose de reference precise a la position (0,0) de la grille). Un
sprite dans un style visuellement tres different peut donner un resultat
degrade (bleeding de couleur, silhouette faussee) — c'est attendu, pas un
bug. Si le resultat n'est pas exploitable, l'etape suivante serait un
fine-tuning sur plusieurs personnages (~15-20) dans le style voulu, pas un
changement de ce script.